In [100]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob
from sft import *
from datetime import datetime, timedelta
from interpolation import interp1d
from utils import *
from scipy.ndimage import gaussian_filter

In [101]:
files = np.array(sorted(glob.glob('/home/ulyanov/data/solo/phi/synoptic_maps/*')))
files

array(['/home/ulyanov/data/solo/phi/synoptic_maps/2279.npz',
       '/home/ulyanov/data/solo/phi/synoptic_maps/2280.npz',
       '/home/ulyanov/data/solo/phi/synoptic_maps/2281.npz',
       '/home/ulyanov/data/solo/phi/synoptic_maps/2282.npz',
       '/home/ulyanov/data/solo/phi/synoptic_maps/2283.npz',
       '/home/ulyanov/data/solo/phi/synoptic_maps/2284.npz',
       '/home/ulyanov/data/solo/phi/synoptic_maps/2285.npz',
       '/home/ulyanov/data/solo/phi/synoptic_maps/2286.npz',
       '/home/ulyanov/data/solo/phi/synoptic_maps/2287.npz',
       '/home/ulyanov/data/solo/phi/synoptic_maps/2288.npz',
       '/home/ulyanov/data/solo/phi/synoptic_maps/2289.npz',
       '/home/ulyanov/data/solo/phi/synoptic_maps/2290.npz',
       '/home/ulyanov/data/solo/phi/synoptic_maps/2291.npz',
       '/home/ulyanov/data/solo/phi/synoptic_maps/2292.npz',
       '/home/ulyanov/data/solo/phi/synoptic_maps/2293.npz',
       '/home/ulyanov/data/solo/phi/synoptic_maps/2294.npz',
       '/home/ulyanov/da

In [102]:
flux = []
weights = []
dates = []

for file in files:
    s = np.load(file)
    mean = s['mean']
    weight = s['weight']
    lati = s['lat']
    start = str(s['start'])

    flux.append(mean)
    weights.append(weight)
    dates.append(datetime.fromisoformat(start))

flux = np.array(flux)
weights = np.array(weights).clip(0,0.01)
flux = np.nanmean(flux * weights, axis=-1) / np.nanmean(weights)
flux = np.nan_to_num(flux)
flux = (flux[:,1:] + flux[:,:-1]) / 2
lat = (lati[1:] + lati[:-1]) / 2

dates = np.array(dates)


/tmp/ipykernel_103600/1184275821.py:18: RuntimeWarning: Mean of empty slice
  flux = np.nanmean(flux * weights, axis=-1) / np.nanmean(weights)


In [28]:
files = np.array(sorted(glob.glob('/home/ulyanov/data/sdo/hmi/synoptic_maps/*')))

flux_hmi = []

for file in files:

    with fits.open(file) as hdul:
        header = hdul[0].header.copy()
        data = hdul[0].data.copy()

    data = rebin(data, 2)
    data = np.nanmean(data, axis=-1)
    data = (data[1:] + data[:-1]) / 2
    data = np.nan_to_num(data)

    flux_hmi.append(data)

flux_hmi = np.array(flux_hmi)

/tmp/ipykernel_103600/2438216199.py:12: RuntimeWarning: Mean of empty slice
  data = np.nanmean(data, axis=-1)


In [103]:
plt.figure(figsize=(10,8))
plt.plot(lat, flux[17])
#plt.plot(lat, flux[23])
plt.plot(lat, flux_hmi[17])
plt.xlim(-90,90)
plt.ylim(-10,10)
plt.grid(True)
plt.tight_layout()

In [91]:
R = 695e8
u = 1000
d = 500e10
lam1 = 45
dt = timedelta(days=1).total_seconds()


xi = lati * np.pi / 180 * R
vi = u * flow(lati, a=0.25)
li = 2 * np.pi * R * np.cos(lati  * np.pi / 180)

y = flux_hmi[0].copy()


Q = []
for t in np.arange(dates[0], dates[23], timedelta(days=1)):
    y_ = interp1d(flux_hmi, dates, t)

    y = advect(y, xi, vi, dt, li)
    y = diffuse(y, xi, d, dt, li)
    y[np.abs(lat) < lam1] = y_[np.abs(lat) < lam1]

    Q.append(y)

Q = np.array(Q)


In [92]:
dates_ = np.arange(dates[0], dates[23], timedelta(days=1))
flux_ = interp1d(flux_hmi, dates, dates_)

plt.figure(figsize=(10,8))
plt.plot(lat, flux_[0])
plt.plot(lat, flux_[-1])
plt.plot(lat, Q[-1])
plt.xlim(-90,90)
plt.ylim(-15,15)
plt.grid(True)
plt.tight_layout()

In [93]:
plt.figure(figsize=(10,8))
plt.imshow(Q.T, 'bwr', aspect='auto', origin='lower', vmin=-10, vmax=10)
plt.tight_layout()

In [94]:
plt.figure(figsize=(10,8))
plt.imshow(flux_.T, 'bwr', aspect='auto', origin='lower', vmin=-10, vmax=10)#, interpolation='bicubic')
plt.tight_layout()

In [106]:
dates[17]

datetime.datetime(2025, 4, 8, 6, 0, 3)